# Aula 12 — Seleção, filtragem e criação de colunas

**Módulo 4 — Pandas e Análise de Dados**

## Objetivos da aula

- Selecionar linhas e colunas específicas de um `DataFrame`.
- Aplicar filtros condicionais para isolar subconjuntos de dados.
- Criar novas colunas a partir de colunas existentes.

---

## 1. Recriando a base de dados

Como cada notebook do curso é independente, começamos recriando a mesma base de monitoramento apresentada na Aula 11 (mesma lógica, mesma semente aleatória — os dados são idênticos).

In [12]:
import numpy as np
import pandas as pd

np.random.seed(42)

n = 500
tipos_equipamento = ["Motor", "Bomba", "Compressor", "Ventilador"]

dados = pd.DataFrame({
    "equipamento_id": [f"EQ-{i:04d}" for i in range(1, n + 1)],
    "tipo_equipamento": np.random.choice(tipos_equipamento, size=n),
    "temperatura": np.round(np.random.normal(70, 12, size=n), 1),
    "pressao": np.round(np.random.normal(5.5, 1.3, size=n), 2),
    "vibracao": np.round(np.random.normal(2.4, 1.1, size=n), 2),
    "horas_operacao": np.random.randint(0, 10000, size=n),
})


def definir_status(linha):
    critico = (linha["temperatura"] >= 90) or (linha["vibracao"] >= 4.5) or (linha["pressao"] >= 8) or (linha["pressao"] <= 2)
    alerta = (linha["temperatura"] >= 80) or (linha["vibracao"] >= 3.5) or (linha["pressao"] >= 7) or (linha["pressao"] <= 3)
    if critico:
        return "critico"
    elif alerta:
        return "alerta"
    else:
        return "normal"


dados["status"] = dados.apply(definir_status, axis=1)
dados.to_csv("sensores_industriais.csv", index=False)

df = pd.read_csv("sensores_industriais.csv")
df.head()


,equipamento_id,tipo_equipamento,temperatura,pressao,vibracao,horas_operacao,status
0,EQ-0001,Compressor,59.8,6.42,3.01,1958,normal
1,EQ-0002,Ventilador,51.8,6.08,1.33,6344,normal
2,EQ-0003,Motor,64.6,5.03,2.52,5779,normal
3,EQ-0004,Compressor,80.3,7.01,0.93,6144,alerta
4,EQ-0005,Compressor,72.6,4.09,1.74,5063,normal


## 2. Selecionando colunas

Uma única coluna é acessada com colchetes e retorna uma `Series`. Várias colunas são selecionadas passando uma **lista** de nomes, o que retorna um novo `DataFrame`.

In [13]:
# uma coluna -> Series
print(type(df["temperatura"]))
df["temperatura"].head()


<class 'pandas.core.series.Series'>


0    59.8
1    51.8
2    64.6
3    80.3
4    72.6
Name: temperatura, dtype: float64

In [14]:
# várias colunas -> DataFrame (repare na lista dupla de colchetes)
df[["equipamento_id", "tipo_equipamento", "temperatura"]].head()


,equipamento_id,tipo_equipamento,temperatura
0,EQ-0001,Compressor,59.8
1,EQ-0002,Ventilador,51.8
2,EQ-0003,Motor,64.6
3,EQ-0004,Compressor,80.3
4,EQ-0005,Compressor,72.6


## 3. Selecionando linhas com `loc` e `iloc`

- `df.loc[rotulo]` seleciona por **rótulo** (nome do índice ou das colunas).
- `df.iloc[posicao]` seleciona por **posição numérica**, como em listas.

In [15]:
print("Linha de índice 0 (loc):")
print(df.loc[0])

print("\nLinhas 0 a 2 e colunas 'temperatura' e 'pressao' (loc):")
print(df.loc[0:2, ["temperatura", "pressao"]])


Linha de índice 0 (loc):
equipamento_id         EQ-0001
tipo_equipamento    Compressor
temperatura               59.8
pressao                   6.42
vibracao                  3.01
horas_operacao            1958
status                  normal
Name: 0, dtype: object

Linhas 0 a 2 e colunas 'temperatura' e 'pressao' (loc):
   temperatura  pressao
0         59.8     6.42
1         51.8     6.08
2         64.6     5.03


In [16]:
print("Primeiras 3 linhas, colunas 2 a 4 (iloc, por posição):")
print(df.iloc[0:3, 2:5])


Primeiras 3 linhas, colunas 2 a 4 (iloc, por posição):
   temperatura  pressao  vibracao
0         59.8     6.42      3.01
1         51.8     6.08      1.33
2         64.6     5.03      2.52


## 4. Filtragem condicional

O padrão mais usado no Pandas: colocar uma **condição** dentro dos colchetes do `DataFrame`. Isso é conceitualmente igual à indexação booleana de arrays NumPy que vimos na Aula 08 — o Pandas usa a mesma ideia sobre tabelas inteiras.

In [17]:
equipamentos_quentes = df[df["temperatura"] >= 90]

print(f"{len(equipamentos_quentes)} equipamentos com temperatura >= 90°C")
equipamentos_quentes.head()


27 equipamentos com temperatura >= 90°C


,equipamento_id,tipo_equipamento,temperatura,pressao,vibracao,horas_operacao,status
21,EQ-0022,Motor,116.2,7.71,4.32,7568,critico
32,EQ-0033,Ventilador,97.8,5.90,1.09,3674,critico
46,EQ-0047,Ventilador,95.7,6.84,1.52,6999,critico
60,EQ-0061,Compressor,91.2,4.71,2.66,1531,critico
64,EQ-0065,Compressor,95.5,4.23,0.21,9335,critico


### Combinando múltiplas condições

Diferente do Python puro, o Pandas usa `&` (e), `|` (ou) e `~` (não) no lugar de `and`, `or`, `not` — e cada condição precisa ficar entre parênteses.

In [18]:
# equipamentos com temperatura alta E pressão alta ao mesmo tempo
criticos_por_dois_fatores = df[(df["temperatura"] >= 85) & (df["pressao"] >= 7)]
print(f"{len(criticos_por_dois_fatores)} equipamentos com temperatura E pressão elevadas")

# equipamentos do tipo Motor OU Compressor
motores_ou_compressores = df[(df["tipo_equipamento"] == "Motor") | (df["tipo_equipamento"] == "Compressor")]
print(f"{len(motores_ou_compressores)} equipamentos são Motor ou Compressor")

# o método .isin() é mais prático que vários 'or' para checar múltiplos valores
motores_ou_compressores_v2 = df[df["tipo_equipamento"].isin(["Motor", "Compressor"])]
print(f"{len(motores_ou_compressores_v2)} equipamentos (com isin)")


8 equipamentos com temperatura E pressão elevadas
244 equipamentos são Motor ou Compressor
244 equipamentos (com isin)


In [19]:
# equipamentos que NÃO estão em status normal
fora_do_normal = df[~(df["status"] == "normal")]
print(f"{len(fora_do_normal)} equipamentos fora do status normal")
fora_do_normal.head()


220 equipamentos fora do status normal


,equipamento_id,tipo_equipamento,temperatura,pressao,vibracao,horas_operacao,status
3,EQ-0004,Compressor,80.3,7.01,0.93,6144,alerta
11,EQ-0012,Compressor,56.3,5.26,3.90,6040,alerta
14,EQ-0015,Ventilador,83.0,4.58,2.53,4628,alerta
15,EQ-0016,Motor,82.6,3.67,3.00,6370,alerta
19,EQ-0020,Compressor,76.2,7.78,1.67,6115,alerta


## 5. Criando novas colunas

Basta atribuir um valor (ou uma expressão calculada a partir de outras colunas) a uma coluna que ainda não existe.

In [20]:
# diferença entre a temperatura de cada equipamento e a média geral da base
df["desvio_temperatura"] = df["temperatura"] - df["temperatura"].mean()

# coluna booleana: equipamento fora do status normal
df["fora_do_normal"] = df["status"] != "normal"

df[["equipamento_id", "temperatura", "desvio_temperatura", "status", "fora_do_normal"]].head()


,equipamento_id,temperatura,desvio_temperatura,status,fora_do_normal
0,EQ-0001,59.8,-10.1202,normal,False
1,EQ-0002,51.8,-18.1202,normal,False
2,EQ-0003,64.6,-5.3202,normal,False
3,EQ-0004,80.3,10.3798,alerta,True
4,EQ-0005,72.6,2.6798,normal,False


### Criando colunas com lógica condicional: `apply` e `np.where`

Para regras mais elaboradas que uma simples conta, usamos `apply()` com uma função (lembra da Aula 06?), ou `np.where()` para condições simples do tipo "se/senão".

In [21]:
def faixa_de_horas(horas):
    if horas < 3000:
        return "baixo uso"
    elif horas < 7000:
        return "uso moderado"
    else:
        return "uso intenso"


df["faixa_uso"] = df["horas_operacao"].apply(faixa_de_horas)

df[["equipamento_id", "horas_operacao", "faixa_uso"]].head()


,equipamento_id,horas_operacao,faixa_uso
0,EQ-0001,1958,baixo uso
1,EQ-0002,6344,uso moderado
2,EQ-0003,5779,uso moderado
3,EQ-0004,6144,uso moderado
4,EQ-0005,5063,uso moderado


In [22]:
import numpy as np

# np.where(condicao, valor_se_true, valor_se_false) -- equivalente vetorizado de um if/else
df["alerta_vibracao"] = np.where(df["vibracao"] >= 3.5, "sim", "não")

df[["equipamento_id", "vibracao", "alerta_vibracao"]].head()


,equipamento_id,vibracao,alerta_vibracao
0,EQ-0001,3.01,não
1,EQ-0002,1.33,não
2,EQ-0003,2.52,não
3,EQ-0004,0.93,não
4,EQ-0005,1.74,não


## 6. Resumo da aula

- `df["coluna"]` retorna uma `Series`; `df[["col1", "col2"]]` retorna um `DataFrame`.
- `loc` seleciona por rótulo, `iloc` seleciona por posição.
- Filtros condicionais usam `&`, `|`, `~` (com parênteses em cada condição) no lugar de `and`, `or`, `not`.
- `.isin([...])` simplifica comparações com múltiplos valores possíveis.
- Novas colunas são criadas por atribuição direta; `apply()` e `np.where()` cobrem lógicas mais elaboradas.

### Exercício sugerido

Crie uma coluna `pressao_fora_da_faixa` que seja `True` quando `pressao < 3` ou `pressao > 7`. Em seguida, filtre o `DataFrame` para mostrar apenas os equipamentos do tipo `"Bomba"` que estão com `pressao_fora_da_faixa` igual a `True`.
